# Ayo Olopon Model


## 1. Load the game

Importing the module registers the game with OpenSpiel under the name
`oware`.

In [13]:
pip install open-spiel 


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import pyspiel
import ayo_olopon  # Registers the game

game = pyspiel.load_game("ayo_olopon")
state = game.new_initial_state()
print(game)
print(state)
print("Legal actions:", state.legal_actions())

ayo_olopon(num_houses_per_player=6,num_seeds_per_house=4)
Board: [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4]
Captured: [0, 0]
Current player: 0
Terminal: False
Legal actions: [0, 1, 2, 3, 4, 5]


## 2. Board representation

The twelve houses are stored in sowing order:

```text
Player 0:  0  1  2  3  4  5
Player 1:  6  7  8  9 10 11
```

Actions are local to a player's row. For example, action `0` refers to
house `0` for player 0 and house `6` for player 1.

In [2]:
print("Board:", state.board)
print("Captured:", state.captured)
print("Current player:", state.current_player())
print("Observation size:", len(state.board) + len(state.captured))

Board: [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4]
Captured: [0, 0]
Current player: 0
Observation size: 14


## 3. Play one move

The selected action is converted to a physical house, its seeds are
distributed counter-clockwise, and the turn changes.

In [3]:
action = state.legal_actions()[0]
print("Playing action:", action)
state.apply_action(action)
print(state)

Playing action: 0
Board: [2, 7, 1, 6, 1, 6, 6, 6, 0, 1, 6, 6]
Captured: [0, 0]
Current player: 1
Terminal: False


## 4. Inspect an observation for a model

The observer returns normalized house counts followed by normalized
captured scores. This is the vector that can be passed to a neural model.

In [4]:
observer = game.make_py_observer()
observer.set_from(state, state.current_player())
print(observer.tensor)
print("Observation shape:", observer.tensor.shape)

[0.04166667 0.14583333 0.02083333 0.125      0.02083333 0.125
 0.125      0.125      0.         0.02083333 0.125      0.125
 0.         0.        ]
Observation shape: (14,)


## 5. Run a random game

This is useful for checking that legal moves, sowing, captures, and
termination work together before connecting a learning algorithm.

In [7]:
random_state = game.new_initial_state()
while not random_state.is_terminal():
    action = random_state.legal_actions()[0]
    random_state.apply_action(action)

print(random_state)
print("Returns:", random_state.returns())

Board: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Captured: [24, 24]
Current player: -4
Terminal: True
Returns: [0.0, 0.0]


## Where to adjust the rules

Edit `Model/oware/oware.py`, especially these methods:

- `_legal_actions` — feeding and legal-move rules
- `_distribute_seeds` — sowing direction and source-house behavior
- `_is_grand_slam` — Grand Slam interpretation
- `_capture_from` — capture rules
- `_collect_and_terminate` — end-of-game scoring